# Recurrent Neural Networks -- Teaching Neurons to Remember

Welcome! This is where things get exciting. So far, our neural networks have had a fundamental limitation: **they have no memory**. Every input is treated independently, as if the network wakes up with amnesia each time. That's fine for classifying a single image, but it's a disaster for anything involving **sequences** -- text, music, stock prices, weather.

## What You'll Learn

By the end of this notebook, you'll understand:
- Why **order matters** and regular neural networks can't handle it
- The **hidden state** -- the key idea that gives RNNs memory
- How to **build an RNN cell from scratch** with NumPy (no frameworks!)
- **Unrolling through time** -- seeing the same cell process a sequence step by step
- **Parameter sharing** -- why the same weights are reused at every step
- A **character prediction** example showing the RNN in action
- How to **visualize hidden states** evolving over a sequence
- A preview of RNN **limitations** (covered in detail in Notebook 02)

**Prerequisites:** Basic understanding of neural networks (what a weight, bias, and activation function are).

---

## Jargon Buster

| Term | Plain English Explanation |
|------|---------------------------|
| **Sequence** | Data where **order matters** -- a sentence, a melody, a series of temperatures |
| **Time step** | One position in the sequence (one word, one note, one day) |
| **Hidden state** | The RNN's "memory" -- a vector summarizing everything it has seen so far |
| **RNN cell** | The tiny processing unit that reads one input and updates the hidden state |
| **Unrolling** | Drawing out the same RNN cell at each time step so you can see the full picture |
| **Parameter sharing** | Using the **same** weights at every time step (the RNN doesn't grow with sequence length) |
| **tanh** | An activation function that squashes values between -1 and +1 (keeps the hidden state bounded) |
| **Vanishing gradient** | The learning signal fading over long sequences -- the core RNN weakness (Notebook 02) |

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

%matplotlib inline

np.random.seed(42)

print("Libraries imported successfully!")
print("NumPy version:", np.__version__)

---

## Part 1: Why Sequences Matter

### The Breakfast Analogy

Imagine you're reading a sentence one word at a time:

> **"The cat sat on the mat"**

After reading "The cat sat", you have a pretty good sense of what's going on. But a regular neural network sees each word **independently** -- it doesn't know that "sat" came after "cat". It's like having amnesia between every single word.

Sequences are everywhere:
- **Language**: "dog bites man" vs "man bites dog" -- same words, completely different meaning!
- **Weather**: Today's temperature depends on yesterday's. A sudden drop means something different than a gradual decline.
- **Music**: The note C after A-B sounds very different from C after E-F. Context is everything.
- **Stock prices**: A price of $100 means nothing without knowing if it was $90 or $110 yesterday.

### Why Regular Neural Networks Fail

A standard feedforward network takes a fixed-size input and produces a fixed-size output. No memory, no notion of "before" or "after".

```
FEEDFORWARD NETWORK (no memory)
================================

  "cat"  -->  [Neural Network]  -->  output
  "sat"  -->  [Neural Network]  -->  output    (completely independent!)
  "on"   -->  [Neural Network]  -->  output    (forgot everything!)

  Each input is processed in isolation.
  The network has NO IDEA what came before.


WHAT WE NEED: a network with MEMORY
=====================================

  "cat"  -->  [Network + Memory]  -->  output   memory: "saw a cat"
                    |
                    v (pass memory forward)
  "sat"  -->  [Network + Memory]  -->  output   memory: "a cat sat"
                    |
                    v (pass memory forward)
  "on"   -->  [Network + Memory]  -->  output   memory: "a cat sat on"

  THIS is an RNN!
```

Let's see this concretely.

In [ ]:
# Demonstrate WHY order matters

print("WHY ORDER MATTERS\n")
print("=" * 70)

# Example 1: Language
print("\n1. LANGUAGE")
print("-" * 40)
s1 = ["dog", "bites", "man"]
s2 = ["man", "bites", "dog"]
print(f"   Sentence A: {' '.join(s1)}")
print(f"   Sentence B: {' '.join(s2)}")
print(f"   Same words: {sorted(s1) == sorted(s2)}")
print(f"   Same meaning: NO! Order changes everything.")

# Example 2: Temperature
print("\n2. WEATHER")
print("-" * 40)
temps_rising  = [60, 65, 70, 75, 80]
temps_falling = [80, 75, 70, 65, 60]
print(f"   Sequence A (warming): {temps_rising}")
print(f"   Sequence B (cooling): {temps_falling}")
print(f"   Same average: {np.mean(temps_rising)} == {np.mean(temps_falling)}")
print(f"   Same trend: NO! One is warming, the other is cooling.")

# Example 3: Music
print("\n3. MUSIC")
print("-" * 40)
print("   C-D-E-F-G  = ascending scale (sounds happy)")
print("   G-F-E-D-C  = descending scale (sounds melancholy)")
print("   Same notes, completely different feeling.")

print("\n" + "=" * 70)
print("\nKEY INSIGHT:")
print("   A bag of words/numbers is NOT the same as a sequence.")
print("   We need a network that processes inputs IN ORDER")
print("   and REMEMBERS what it has seen so far.")
print("=" * 70)

---

## Part 2: The Key Idea -- Hidden State

### The Book Analogy

Think about how **you** read a book:

1. You read **one page at a time** (you can't see all pages simultaneously)
2. As you read each page, you **update your mental summary** of the story
3. Your understanding of page 50 depends on everything you read on pages 1-49
4. You don't memorize every word -- you keep a **compressed summary**

That compressed mental summary? That's exactly what an RNN calls the **hidden state**.

```
THE HIDDEN STATE = YOUR MENTAL SUMMARY
========================================

  Page 1:  "Once upon a time, there was a king."
           Mental summary: "Story about a king"
                            |
                            v
  Page 2:  "The king had a daughter who loved dragons."
           Mental summary: "King has a dragon-loving daughter"
                            |
                            v
  Page 3:  "One day, a dragon appeared at the castle."
           Mental summary: "Dragon arrived, daughter will probably be happy"

  Your brain doesn't store every word.
  It keeps a COMPRESSED SUMMARY that updates with each new input.
  That's the hidden state!
```

### The Math (Don't Panic!)

The hidden state $h_t$ at time step $t$ is computed as:

$$h_t = \tanh(W_{hh} \cdot h_{t-1} + W_{xh} \cdot x_t + b_h)$$

Let's break this down piece by piece:

| Symbol | What It Is | Plain English |
|--------|-----------|---------------|
| $h_t$ | Hidden state at time $t$ | Your mental summary after reading page $t$ |
| $h_{t-1}$ | Hidden state from the previous step | Your mental summary from the previous page |
| $x_t$ | Input at time $t$ | The current page you're reading |
| $W_{hh}$ | Hidden-to-hidden weights | How to combine the old summary with new info |
| $W_{xh}$ | Input-to-hidden weights | How to extract important info from the current input |
| $b_h$ | Bias | A slight nudge (like all neural network biases) |
| $\tanh$ | Activation function | Squashes the result between -1 and +1 |

In words: **new memory = tanh(transform old memory + transform new input + bias)**

In [ ]:
# Visualize the tanh activation function
# (It's the "squisher" that keeps the hidden state bounded)

x_vals = np.linspace(-5, 5, 200)
y_vals = np.tanh(x_vals)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_vals, y_vals, 'b-', linewidth=3, label='tanh(x)')
ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
ax.axhline(y=1, color='red', linestyle='--', linewidth=1, alpha=0.5, label='Upper bound: +1')
ax.axhline(y=-1, color='red', linestyle='--', linewidth=1, alpha=0.5, label='Lower bound: -1')
ax.axvline(x=0, color='gray', linestyle='-', linewidth=0.5)
ax.set_xlabel('Input', fontsize=12)
ax.set_ylabel('Output', fontsize=12)
ax.set_title('tanh Activation: Squashes any value to [-1, +1]', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_ylim(-1.5, 1.5)
plt.tight_layout()
plt.show()

print("Why tanh?")
print("  It keeps the hidden state bounded between -1 and +1.")
print("  Without it, the hidden state could explode to infinity!")
print(f"  tanh(0)    = {np.tanh(0):.2f}   (zero in, zero out)")
print(f"  tanh(2)    = {np.tanh(2):.4f}  (large input, squashed close to 1)")
print(f"  tanh(100)  = {np.tanh(100):.4f}  (huge input, still just 1.0)")
print(f"  tanh(-3)   = {np.tanh(-3):.4f} (negative input, squashed close to -1)")

---

## Part 3: RNN Cell From Scratch

Now let's build the actual RNN cell. We'll go **step by step**, showing every matrix multiplication so there's no mystery.

### The Recipe

```
AN RNN CELL: THE BUILDING BLOCK
=================================

      h_{t-1}          x_t
  (old memory)    (new input)
        |               |
        v               v
   [ W_hh * h ]   [ W_xh * x ]
        |               |
        +-------+-------+
                |
            + b_h (bias)
                |
            tanh( )
                |
                v
              h_t
          (new memory)

  That's it! One tiny unit. But it's all you need.
```

In [ ]:
# Step-by-step RNN cell computation
# Let's use TINY dimensions so we can see every number

print("STEP-BY-STEP RNN CELL COMPUTATION")
print("=" * 60)

# Dimensions: input_size=2, hidden_size=3
# (tiny, so we can print everything)

input_size = 2
hidden_size = 3

# Initialize weights (small values for clarity)
np.random.seed(7)
W_xh = np.round(np.random.randn(hidden_size, input_size) * 0.5, 2)
W_hh = np.round(np.random.randn(hidden_size, hidden_size) * 0.5, 2)
b_h  = np.round(np.random.randn(hidden_size, 1) * 0.1, 2)

print(f"\nInput size:  {input_size}")
print(f"Hidden size: {hidden_size}")

print(f"\n--- Weights ---")
print(f"W_xh (input -> hidden):    shape {W_xh.shape}")
print(W_xh)
print(f"\nW_hh (hidden -> hidden):   shape {W_hh.shape}")
print(W_hh)
print(f"\nb_h (bias):                shape {b_h.shape}")
print(b_h.flatten())

# Inputs
h_prev = np.array([[0.0], [0.0], [0.0]])  # initial hidden state (all zeros)
x_t = np.array([[1.0], [0.5]])             # current input

print(f"\n--- Inputs ---")
print(f"h_prev (old memory):  {h_prev.flatten()}  (starts at zero!)")
print(f"x_t (current input):  {x_t.flatten()}")

# Step 1: W_xh * x_t
step1 = W_xh @ x_t
print(f"\n--- Step 1: W_xh @ x_t ---")
print(f"   {W_xh.shape} @ {x_t.shape} = {step1.shape}")
print(f"   Result: {step1.flatten()}")

# Step 2: W_hh * h_prev
step2 = W_hh @ h_prev
print(f"\n--- Step 2: W_hh @ h_prev ---")
print(f"   {W_hh.shape} @ {h_prev.shape} = {step2.shape}")
print(f"   Result: {step2.flatten()}  (zeros, because h_prev is zero!)")

# Step 3: Add them together + bias
step3 = step1 + step2 + b_h
print(f"\n--- Step 3: Add everything ---")
print(f"   W_xh @ x_t + W_hh @ h_prev + b_h")
print(f"   {step1.flatten()} + {step2.flatten()} + {b_h.flatten()}")
print(f"   = {step3.flatten()}")

# Step 4: Apply tanh
h_t = np.tanh(step3)
print(f"\n--- Step 4: Apply tanh ---")
print(f"   tanh({np.round(step3.flatten(), 4)})")
print(f"   = {np.round(h_t.flatten(), 4)}")

print(f"\n" + "=" * 60)
print(f"NEW HIDDEN STATE h_t = {np.round(h_t.flatten(), 4)}")
print(f"\nThis is the RNN's 'memory' after seeing the first input!")
print(f"Notice: every value is between -1 and +1 (thanks to tanh).")
print("=" * 60)

In [ ]:
# Now let's wrap this into a clean class

class RNNCell:
    """
    A single vanilla RNN cell built from scratch.
    
    At each time step it takes:
      - x_t:    the current input
      - h_prev: the previous hidden state (memory)
    
    And produces:
      - h_t: the new hidden state (updated memory)
    
    Formula: h_t = tanh(W_xh @ x_t + W_hh @ h_prev + b_h)
    """
    
    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        # Xavier initialization (keeps values stable)
        scale_xh = np.sqrt(2.0 / (input_size + hidden_size))
        scale_hh = np.sqrt(2.0 / (hidden_size + hidden_size))
        
        self.W_xh = np.random.randn(hidden_size, input_size) * scale_xh
        self.W_hh = np.random.randn(hidden_size, hidden_size) * scale_hh
        self.b_h  = np.zeros((hidden_size, 1))
    
    def forward_step(self, x_t, h_prev):
        """
        Process ONE time step.
        
        Parameters:
          x_t:    current input, shape (input_size, 1)
          h_prev: previous hidden state, shape (hidden_size, 1)
        
        Returns:
          h_t: new hidden state, shape (hidden_size, 1)
        """
        h_t = np.tanh(self.W_xh @ x_t + self.W_hh @ h_prev + self.b_h)
        return h_t
    
    def forward_sequence(self, X):
        """
        Process an ENTIRE sequence, one step at a time.
        
        Parameters:
          X: input sequence, shape (seq_len, input_size)
        
        Returns:
          hidden_states: all hidden states, shape (seq_len, hidden_size)
        """
        seq_len = X.shape[0]
        h = np.zeros((self.hidden_size, 1))  # start with blank memory
        hidden_states = []
        
        for t in range(seq_len):
            x_t = X[t].reshape(-1, 1)
            h = self.forward_step(x_t, h)
            hidden_states.append(h.flatten())
        
        return np.array(hidden_states)


# Quick test
np.random.seed(42)
rnn = RNNCell(input_size=2, hidden_size=3)

# A sequence of 4 inputs, each with 2 features
X_test = np.random.randn(4, 2)
states = rnn.forward_sequence(X_test)

print("RNNCell class defined!")
print(f"\nTest: {X_test.shape[0]} time steps, input_size={X_test.shape[1]}, hidden_size=3")
print(f"Hidden states shape: {states.shape}  (seq_len, hidden_size)")
print(f"\nHidden state at each step:")
for t in range(len(states)):
    print(f"  t={t}: {np.round(states[t], 4)}")

---

## Part 4: Unrolling Through Time

Here's a crucial concept. An RNN is really just **one cell** that gets reused over and over. But to understand what's happening, we "unroll" it -- we draw a separate copy for each time step.

```
ROLLED UP (compact view)         UNROLLED (what actually happens)
=========================        =====================================

     +-------+                    +-------+   +-------+   +-------+
     |       |                    |       |   |       |   |       |
x -->| RNN   |--> h        x_0 ->| RNN   |-->| RNN   |-->| RNN   |--> h_3
     | Cell  |                    | Cell  |   | Cell  |   | Cell  |
     |       |                    |       |   |       |   |       |
     +---^---+                    +-------+   +-------+   +-------+
         |                            ^           ^           ^
         +-- (loops back)            x_0         x_1         x_2

  Same cell!                    Same weights at every step!
  Processes one input           But drawn out so we can see
  at a time.                    the flow of information.
```

**Key insight**: The rolled and unrolled views show the **exact same computation**. Unrolling is just a way to visualize it.

In [ ]:
# Visualize unrolling: show the same RNN cell processing a sequence step by step

fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(-1, 13)
ax.set_ylim(-1, 5.5)
ax.axis('off')

words = ["The", "cat", "sat", "on", "the"]
n = len(words)
x_positions = [1.0 + i * 2.5 for i in range(n)]

y_input = 0.5
y_cell = 2.5
y_output = 4.5

# Draw input words
for i, (x, word) in enumerate(zip(x_positions, words)):
    ax.text(x, y_input, word, ha='center', va='center', fontsize=13,
            fontweight='bold', bbox=dict(boxstyle='round,pad=0.3',
            facecolor='lightyellow', edgecolor='orange', linewidth=2))
    # Arrow from input to cell
    ax.annotate('', xy=(x, y_cell - 0.4), xytext=(x, y_input + 0.3),
                arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'))
    ax.text(x + 0.15, (y_input + y_cell) / 2, f'$x_{i}$', fontsize=10, color='gray')

# Draw RNN cells
for i, x in enumerate(x_positions):
    rect = plt.Rectangle((x - 0.5, y_cell - 0.4), 1.0, 0.8,
                          facecolor='#AED6F1', edgecolor='#2980B9',
                          linewidth=2.5, zorder=3)
    ax.add_patch(rect)
    ax.text(x, y_cell, 'RNN\nCell', ha='center', va='center',
            fontsize=10, fontweight='bold', color='#2980B9')

# Draw hidden state arrows between cells
# Initial h_0
ax.annotate('', xy=(x_positions[0] - 0.5, y_cell),
            xytext=(x_positions[0] - 1.3, y_cell),
            arrowprops=dict(arrowstyle='->', lw=2.5, color='#E74C3C'))
ax.text(x_positions[0] - 1.6, y_cell, '$h_0$\n(zeros)', ha='center', va='center',
        fontsize=9, color='#E74C3C', fontweight='bold')

for i in range(n - 1):
    ax.annotate('', xy=(x_positions[i+1] - 0.5, y_cell),
                xytext=(x_positions[i] + 0.5, y_cell),
                arrowprops=dict(arrowstyle='->', lw=2.5, color='#E74C3C'))
    ax.text((x_positions[i] + x_positions[i+1]) / 2, y_cell + 0.55,
            f'$h_{i+1}$', ha='center', va='center', fontsize=11,
            color='#E74C3C', fontweight='bold')

# Final hidden state arrow
ax.annotate('', xy=(x_positions[-1] + 1.3, y_cell),
            xytext=(x_positions[-1] + 0.5, y_cell),
            arrowprops=dict(arrowstyle='->', lw=2.5, color='#E74C3C'))
ax.text(x_positions[-1] + 1.6, y_cell, f'$h_{n}$', ha='center', va='center',
        fontsize=12, color='#E74C3C', fontweight='bold')

# "Same weights" label
ax.annotate('', xy=(x_positions[0], y_cell + 0.9),
            xytext=(x_positions[-1], y_cell + 0.9),
            arrowprops=dict(arrowstyle='<->', lw=1.5, color='green'))
ax.text((x_positions[0] + x_positions[-1]) / 2, y_cell + 1.2,
        'SAME weights (W_xh, W_hh, b_h) reused at every step!',
        ha='center', va='center', fontsize=11, fontweight='bold', color='green')

# Labels
ax.text(-0.8, y_input, 'Input\nWords', ha='center', va='center',
        fontsize=11, fontweight='bold', color='orange')
ax.text(-0.8, y_cell, 'RNN\nCells', ha='center', va='center',
        fontsize=11, fontweight='bold', color='#2980B9')

ax.set_title('RNN Unrolled Through Time: Same Cell, Reused at Every Step',
             fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print("\nWhat you're seeing:")
print("  - Yellow boxes  = input words at each time step")
print("  - Blue boxes    = the RNN cell (same one reused!)")
print("  - Red arrows    = hidden state flowing from step to step")
print("  - Green bracket = all cells share the SAME weights")

In [ ]:
# Watch the hidden state evolve step by step through a sequence

print("WATCHING THE HIDDEN STATE EVOLVE\n")
print("=" * 60)

np.random.seed(42)
rnn_demo = RNNCell(input_size=2, hidden_size=3)

# Simple sequence: 5 time steps
sequence = np.array([
    [1.0, 0.0],   # input at t=0
    [0.0, 1.0],   # input at t=1
    [1.0, 1.0],   # input at t=2
    [0.5, 0.5],   # input at t=3
    [0.0, 0.0],   # input at t=4 (no input!)
])

h = np.zeros((3, 1))  # start with blank memory
print(f"Initial hidden state h_0 = {h.flatten()}  (blank memory)\n")

for t in range(len(sequence)):
    x_t = sequence[t].reshape(-1, 1)
    h = rnn_demo.forward_step(x_t, h)
    print(f"Step t={t}: input = {sequence[t]}")
    print(f"          h_{t+1}   = {np.round(h.flatten(), 4)}")
    print(f"          (values between -1 and +1, as expected)")
    print()

print("=" * 60)
print("\nNotice:")
print("  - h_0 starts at all zeros (blank memory)")
print("  - Each step CHANGES the hidden state based on the new input")
print("  - At t=4 (input=[0,0]), the hidden state STILL changes")
print("    because it also depends on the PREVIOUS hidden state")
print("  - The RNN remembers what it saw before!")
print("=" * 60)

---

## Part 5: Parameter Sharing

This is one of the most elegant ideas in RNNs. **The exact same weights are used at every time step.**

### Why Is This Powerful?

1. **Handles any sequence length**: The same cell works whether your sequence is 5 words or 500 words. No redesign needed.

2. **Fewer parameters**: A feedforward network processing a 100-word sentence would need 100 separate sets of weights. An RNN needs just one set.

3. **Generalization**: If the network learns that "cat sat" is a valid pattern, it recognizes it whether it appears at position 1 or position 99.

```
FEEDFORWARD (no sharing)         RNN (parameter sharing)
===========================      ===========================

  x_0 -> [W_0] -> output_0      x_0 -> [W] -> h_1
  x_1 -> [W_1] -> output_1      x_1 -> [W] -> h_2    (same W!)
  x_2 -> [W_2] -> output_2      x_2 -> [W] -> h_3    (same W!)
  ...                            ...
  x_99-> [W_99]-> output_99     x_99 -> [W] -> h_100  (STILL same W!)

  100 different weight           1 set of weights,
  matrices!                      reused 100 times!

  Can't handle length 101       Works for ANY length!
  without redesign.
```

This is conceptually similar to how CNNs share filter weights across spatial positions -- but here, we share across **time** positions.

In [ ]:
# Demonstrate parameter sharing: same RNN handles different sequence lengths

print("PARAMETER SHARING IN ACTION\n")
print("=" * 60)

np.random.seed(42)
rnn_shared = RNNCell(input_size=3, hidden_size=4)

# Count the parameters
n_params_xh = rnn_shared.W_xh.size
n_params_hh = rnn_shared.W_hh.size
n_params_b  = rnn_shared.b_h.size
total_params = n_params_xh + n_params_hh + n_params_b

print(f"RNN parameters (input_size=3, hidden_size=4):")
print(f"  W_xh: {rnn_shared.W_xh.shape} = {n_params_xh} parameters")
print(f"  W_hh: {rnn_shared.W_hh.shape} = {n_params_hh} parameters")
print(f"  b_h:  {rnn_shared.b_h.shape}   = {n_params_b} parameters")
print(f"  TOTAL: {total_params} parameters")

print(f"\nNow let's use the SAME {total_params} parameters on different lengths:\n")

for seq_len in [3, 10, 50, 200]:
    X = np.random.randn(seq_len, 3)
    states = rnn_shared.forward_sequence(X)
    print(f"  Sequence length {seq_len:>3}: processed successfully!  "
          f"Output shape: {states.shape}  "
          f"(still just {total_params} parameters)")

print(f"\n" + "=" * 60)
print(f"KEY POINT: {total_params} parameters handle ANY sequence length.")
print(f"A feedforward net would need {total_params} x seq_len parameters!")
print("=" * 60)

---

## Part 6: Simple Example -- Character Prediction

Let's put our RNN to work on a concrete task: predicting the next character in a sequence. We'll feed it the characters of a short word and watch what happens inside.

**Task**: Feed the word "hello" one character at a time. At each step, see how the hidden state changes.

We won't train the RNN here (that's Notebook 02!). Instead, we'll focus on understanding the **forward pass** -- how information flows through the network.

```
CHARACTER PREDICTION SETUP
===========================

  Input:   h -> e -> l -> l -> o
           |    |    |    |    |
           v    v    v    v    v
          RNN  RNN  RNN  RNN  RNN
           |    |    |    |    |
  Hidden: h_1  h_2  h_3  h_4  h_5
           |    |    |    |    |
           v    v    v    v    v
  Output:  ?    ?    ?    ?    ?   (predict next character)

  We encode each character as a one-hot vector.
```

In [ ]:
# Character prediction: encoding characters and running through RNN

print("CHARACTER PREDICTION EXAMPLE\n")
print("=" * 60)

# Our tiny vocabulary
word = "hello"
chars = sorted(set(word))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}
vocab_size = len(chars)

print(f"Word: '{word}'")
print(f"Unique characters: {chars}")
print(f"Vocabulary size: {vocab_size}")
print(f"Character to index mapping: {char_to_idx}")

# One-hot encode each character
def one_hot(char, char_to_idx, vocab_size):
    vec = np.zeros(vocab_size)
    vec[char_to_idx[char]] = 1.0
    return vec

print(f"\nOne-hot encodings:")
for ch in chars:
    print(f"  '{ch}' -> {one_hot(ch, char_to_idx, vocab_size)}")

# Create input sequence
X_chars = np.array([one_hot(ch, char_to_idx, vocab_size) for ch in word])
print(f"\nInput sequence shape: {X_chars.shape} (seq_len={len(word)}, vocab_size={vocab_size})")
print("=" * 60)

In [ ]:
# Run through the RNN and watch the hidden state at each character

np.random.seed(42)
hidden_size_char = 8  # 8 hidden units
rnn_char = RNNCell(input_size=vocab_size, hidden_size=hidden_size_char)

# Also create an output layer: hidden state -> character probabilities
W_hy = np.random.randn(vocab_size, hidden_size_char) * 0.3  # hidden -> output
b_y  = np.zeros((vocab_size, 1))

def softmax(z):
    """Convert raw scores to probabilities (sums to 1)."""
    e_z = np.exp(z - np.max(z))
    return e_z / e_z.sum()

print("FEEDING 'hello' THROUGH THE RNN\n")
print("=" * 60)

h = np.zeros((hidden_size_char, 1))
all_hidden = []
all_probs = []

for t, ch in enumerate(word):
    x_t = X_chars[t].reshape(-1, 1)
    h = rnn_char.forward_step(x_t, h)
    all_hidden.append(h.flatten())
    
    # Compute output probabilities
    y_raw = W_hy @ h + b_y
    probs = softmax(y_raw).flatten()
    all_probs.append(probs)
    
    predicted_idx = np.argmax(probs)
    predicted_char = idx_to_char[predicted_idx]
    
    print(f"Step t={t}: input='{ch}'")
    print(f"  Hidden state: [{', '.join(f'{v:.3f}' for v in h.flatten()[:4])}...]")
    print(f"  Predictions:  ", end="")
    for i, c in enumerate(chars):
        marker = " <--" if i == predicted_idx else ""
        print(f"'{c}'={probs[i]:.3f}{marker}  ", end="")
    print(f"\n  Predicted next: '{predicted_char}'")
    print()

all_hidden = np.array(all_hidden)
all_probs = np.array(all_probs)

print("=" * 60)
print("\nNote: The predictions are random because we haven't trained")
print("the weights yet! The point is to see how the hidden state")
print("changes at each step -- that's the RNN's evolving memory.")
print("Training comes in Notebook 02.")
print("=" * 60)

---

## Part 7: Visualizing Hidden States

The hidden state is the heart of the RNN -- it's where the "memory" lives. Let's visualize how it changes as the RNN reads each character.

In [ ]:
# Heatmap of hidden states over time

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Hidden state heatmap
im1 = axes[0].imshow(all_hidden.T, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_xticks(range(len(word)))
axes[0].set_xticklabels([f"'{c}'" for c in word], fontsize=12)
axes[0].set_xlabel('Input Character', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Hidden Unit', fontsize=12, fontweight='bold')
axes[0].set_yticks(range(hidden_size_char))
axes[0].set_title('Hidden State at Each Step', fontsize=14, fontweight='bold')
plt.colorbar(im1, ax=axes[0], shrink=0.8, label='Activation')

# Plot 2: Output probabilities
im2 = axes[1].imshow(all_probs.T, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
axes[1].set_xticks(range(len(word)))
axes[1].set_xticklabels([f"'{c}'" for c in word], fontsize=12)
axes[1].set_xlabel('Input Character', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Output Character', fontsize=12, fontweight='bold')
axes[1].set_yticks(range(vocab_size))
axes[1].set_yticklabels([f"'{c}'" for c in chars], fontsize=12)
axes[1].set_title('Predicted Next Character Probabilities', fontsize=14, fontweight='bold')
plt.colorbar(im2, ax=axes[1], shrink=0.8, label='Probability')

plt.suptitle('RNN Processing "hello" -- Hidden States and Predictions',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("What you're seeing:")
print("  LEFT:  Each column is the hidden state when the RNN reads that character.")
print("         Red = positive activation, Blue = negative. Notice how it CHANGES")
print("         at every step -- the memory is being updated!")
print("  RIGHT: The RNN's guess for the next character. (Random since untrained.)")

In [ ]:
# Line plot: track individual hidden units over time

fig, ax = plt.subplots(figsize=(10, 5))

colors = plt.cm.tab10(np.linspace(0, 1, hidden_size_char))
for unit in range(hidden_size_char):
    ax.plot(range(len(word)), all_hidden[:, unit], 'o-',
            color=colors[unit], linewidth=2, markersize=8,
            label=f'Unit {unit}')

ax.set_xticks(range(len(word)))
ax.set_xticklabels([f"'{c}'\n(t={i})" for i, c in enumerate(word)], fontsize=11)
ax.set_xlabel('Input Character (Time Step)', fontsize=12, fontweight='bold')
ax.set_ylabel('Hidden State Value', fontsize=12, fontweight='bold')
ax.set_title('Hidden Units Evolving Over Time', fontsize=14, fontweight='bold')
ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
ax.set_ylim(-1.1, 1.1)
ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("What you're seeing:")
print("  Each colored line is one 'neuron' in the hidden state.")
print("  Watch how they change as each character is read:")
print("  - Some neurons respond strongly to certain characters")
print("  - Some stay stable (they might encode long-term patterns)")
print("  - The COMBINATION of all neurons = the RNN's complete memory")

In [ ]:
# Let's try a longer sequence to see how the hidden state evolves over time

np.random.seed(42)

long_text = "the cat sat on the mat and the dog sat on the rug"
long_chars = sorted(set(long_text))
long_char_to_idx = {ch: i for i, ch in enumerate(long_chars)}
long_vocab_size = len(long_chars)

# One-hot encode
X_long = np.array([one_hot(ch, long_char_to_idx, long_vocab_size) for ch in long_text])

# Run through RNN
rnn_long = RNNCell(input_size=long_vocab_size, hidden_size=16)
long_states = rnn_long.forward_sequence(X_long)

# Plot heatmap
fig, ax = plt.subplots(figsize=(16, 5))
im = ax.imshow(long_states.T, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(long_text)))
ax.set_xticklabels(list(long_text), fontsize=7, fontfamily='monospace')
ax.set_xlabel('Characters in Sequence', fontsize=12, fontweight='bold')
ax.set_ylabel('Hidden Unit', fontsize=12, fontweight='bold')
ax.set_title(f'Hidden States for: "{long_text}"', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.6, label='Activation')
plt.tight_layout()
plt.show()

print(f"Sequence length: {len(long_text)} characters")
print(f"Hidden size: 16 units")
print(f"\nNotice the PATTERNS in the heatmap:")
print("  - Spaces create distinctive vertical bands (the RNN 'notices' word boundaries)")
print("  - Similar characters produce similar hidden state patterns")
print("  - The hidden state smoothly evolves -- it doesn't jump randomly")
print("  - This is the RNN's 'memory' being continuously updated!")

---

## Part 8: What About Different Hidden Sizes?

The **hidden size** is a crucial choice when designing an RNN. It controls how much "memory capacity" the network has:

- **Small hidden size** (e.g., 4): Can only remember a little. Like trying to summarize a book in 4 words.
- **Large hidden size** (e.g., 512): Can remember much more. Like having a full notebook to write your summary in.

But bigger isn't always better -- more parameters means more data needed for training, and more computation.

Let's see the effect.

In [ ]:
# Compare different hidden sizes on the same input

np.random.seed(42)

# Use a simple repeating sequence
seq_len_demo = 30
input_dim = 3
# Repeating pattern: [1,0,0], [0,1,0], [0,0,1], [1,0,0], ...
X_pattern = np.array([[1,0,0], [0,1,0], [0,0,1]] * 10)

hidden_sizes_to_test = [2, 8, 32]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax_idx, hs in enumerate(hidden_sizes_to_test):
    np.random.seed(42)
    rnn_test = RNNCell(input_size=input_dim, hidden_size=hs)
    states = rnn_test.forward_sequence(X_pattern)
    
    n_params = rnn_test.W_xh.size + rnn_test.W_hh.size + rnn_test.b_h.size
    
    im = axes[ax_idx].imshow(states.T, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
    axes[ax_idx].set_xlabel('Time Step', fontsize=11)
    axes[ax_idx].set_ylabel('Hidden Unit', fontsize=11)
    axes[ax_idx].set_title(f'Hidden Size = {hs}\n({n_params} params)',
                           fontsize=13, fontweight='bold')
    plt.colorbar(im, ax=axes[ax_idx], shrink=0.7)

plt.suptitle('Effect of Hidden Size on Memory Capacity',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("What you're seeing:")
print("  - Hidden size 2:  Very limited memory (only 2 neurons to encode everything)")
print("  - Hidden size 8:  More capacity, can capture the repeating pattern better")
print("  - Hidden size 32: Lots of capacity, many different internal representations")
print("\nThe input is a repeating pattern [1,0,0] -> [0,1,0] -> [0,0,1] -> ...")
print("Larger hidden sizes can encode this pattern in richer ways.")

---

## Part 9: Limitations Preview -- Why Vanilla RNNs Struggle

Our simple RNN works, but it has a **serious weakness** that becomes apparent with long sequences.

### The Telephone Game Problem

Remember the telephone game? You whisper a message through a chain of people, and by the end it's completely garbled. That's essentially what happens in an RNN:

```
THE VANISHING GRADIENT PROBLEM (preview)
==========================================

  Short sequence (works fine):

    x_0 --> [RNN] --> [RNN] --> [RNN] --> output
                                          ^
    Information from x_0 survives          |
    just 3 steps -- no problem!       "I remember x_0!"


  Long sequence (trouble!):

    x_0 --> [RNN] --> ... (50 cells) ... --> [RNN] --> output
                                                       ^
    Information from x_0 must survive                   |
    50 transformations -- it fades!              "x_0? What x_0?"


  Like the telephone game:
  After enough people, the original message is lost.
```

This is called the **vanishing gradient problem**: during training, the learning signal (gradient) shrinks exponentially as it travels back through many time steps. The network essentially "forgets" old inputs.

**The fix?** LSTM and GRU cells (Notebooks 03 and 04) add special "gates" that control what to remember and what to forget, allowing information to survive across much longer sequences.

Let's see the forgetting problem in action.

In [ ]:
# Demonstrate the "forgetting" problem
# Feed a distinctive signal at t=0, then see how quickly the RNN forgets it

np.random.seed(42)

input_size_demo = 4
hidden_size_demo = 8
rnn_forget = RNNCell(input_size=input_size_demo, hidden_size=hidden_size_demo)

# Create a sequence: strong signal at t=0, then noise
seq_len_long = 50
X_forget = np.random.randn(seq_len_long, input_size_demo) * 0.1  # small noise
X_forget[0] = [5.0, -5.0, 5.0, -5.0]  # STRONG distinctive signal at t=0

states_forget = rnn_forget.forward_sequence(X_forget)

# Measure how "different" the hidden state is from the noise-only baseline
# Run the same RNN with just noise (no signal at t=0)
X_noise_only = np.random.randn(seq_len_long, input_size_demo) * 0.1
states_noise = rnn_forget.forward_sequence(X_noise_only)

# Compute distance between the two hidden states at each time step
distances = np.linalg.norm(states_forget - states_noise, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Hidden states over time (signal version)
for unit in range(hidden_size_demo):
    axes[0].plot(states_forget[:, unit], alpha=0.6, linewidth=1.5)
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Signal at t=0')
axes[0].set_xlabel('Time Step', fontsize=12)
axes[0].set_ylabel('Hidden State Value', fontsize=12)
axes[0].set_title('Hidden State After a Strong Signal at t=0', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Plot 2: Memory decay
axes[1].plot(distances, 'o-', color='#E74C3C', linewidth=2, markersize=3)
axes[1].axhline(y=0, color='gray', linestyle='--', linewidth=0.5)
axes[1].set_xlabel('Time Step', fontsize=12)
axes[1].set_ylabel('"Memory" of Signal at t=0', fontsize=12)
axes[1].set_title('How Quickly the RNN Forgets', fontsize=13, fontweight='bold')
axes[1].annotate('Strong signal here!', xy=(0, distances[0]),
                 xytext=(10, distances[0] * 0.8),
                 arrowprops=dict(arrowstyle='->', color='red', lw=2),
                 fontsize=11, color='red', fontweight='bold')
axes[1].annotate('Mostly forgotten...', xy=(40, distances[40]),
                 xytext=(30, distances[0] * 0.4),
                 arrowprops=dict(arrowstyle='->', color='gray', lw=1.5),
                 fontsize=11, color='gray')
axes[1].grid(alpha=0.3)

plt.suptitle('The RNN Forgetting Problem', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("What you're seeing:")
print(f"  At t=0, we gave the RNN a strong signal: {X_forget[0]}")
print(f"  The right plot shows how much of that signal survives over time.")
print(f"  Memory at t=0:  {distances[0]:.4f} (strong!)")
print(f"  Memory at t=10: {distances[10]:.4f}")
print(f"  Memory at t=30: {distances[30]:.4f}")
print(f"  Memory at t=49: {distances[49]:.4f}")
print(f"\nThe signal fades rapidly! This is why vanilla RNNs struggle")
print(f"with long-range dependencies. LSTMs and GRUs fix this (Notebooks 03-04).")

---

## Part 10: Summary

### Visual Cheat Sheet

```
RNN FUNDAMENTALS -- CHEAT SHEET
================================

1. THE PROBLEM:
   Regular neural networks have no memory.
   They can't handle sequences where ORDER matters.

2. THE SOLUTION:
   RNNs have a HIDDEN STATE that acts as memory.
   h_t = tanh(W_xh * x_t + W_hh * h_{t-1} + b_h)

3. THE ARCHITECTURE:
           h_0     h_1     h_2     h_3
    x_0 -> [RNN] -> [RNN] -> [RNN] -> [RNN] -> h_4

   Same cell, same weights at every step!

4. KEY PROPERTIES:
   - Parameter sharing:   same weights for all time steps
   - Variable length:     works for sequences of any length
   - Hidden state = memory: compressed summary of everything seen

5. THE WEAKNESS:
   Vanishing gradients: forgets old inputs in long sequences.
   Fix: LSTM and GRU cells (Notebooks 03 and 04).
```

### Key Takeaways

1. **Sequences need memory**: Text, audio, time series -- order matters, and regular networks can't handle it.

2. **The hidden state is the key idea**: A vector that gets updated at every time step, carrying forward a compressed summary of everything the network has seen.

3. **One formula does it all**: $h_t = \tanh(W_{xh} \cdot x_t + W_{hh} \cdot h_{t-1} + b_h)$. Just two matrix multiplications, an addition, and a tanh.

4. **Parameter sharing is elegant**: The same weights work at every position, making the model compact and able to handle any sequence length.

5. **Vanilla RNNs forget**: Over long sequences, the hidden state loses information about early inputs. This motivates LSTM and GRU architectures.

In [ ]:
# Final summary visualization: the RNN at a glance

fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(-1, 15)
ax.set_ylim(-1.5, 7)
ax.axis('off')

# Title
ax.text(7, 6.5, 'RNN Fundamentals -- The Big Picture',
        ha='center', va='center', fontsize=18, fontweight='bold')

# Box 1: The Problem
rect1 = mpatches.FancyBboxPatch((0, 4.5), 4, 1.5, boxstyle='round,pad=0.1',
                                 facecolor='#FADBD8', edgecolor='#E74C3C', linewidth=2)
ax.add_patch(rect1)
ax.text(2, 5.6, 'THE PROBLEM', ha='center', va='center', fontsize=12,
        fontweight='bold', color='#E74C3C')
ax.text(2, 5.0, 'Regular networks\nhave no memory.\nOrder is lost.', ha='center', va='center', fontsize=10)

# Arrow
ax.annotate('', xy=(5.5, 5.25), xytext=(4.2, 5.25),
            arrowprops=dict(arrowstyle='->', lw=3, color='gray'))

# Box 2: The Solution
rect2 = mpatches.FancyBboxPatch((5.5, 4.5), 4, 1.5, boxstyle='round,pad=0.1',
                                 facecolor='#D5F5E3', edgecolor='#27AE60', linewidth=2)
ax.add_patch(rect2)
ax.text(7.5, 5.6, 'THE SOLUTION: RNN', ha='center', va='center', fontsize=12,
        fontweight='bold', color='#27AE60')
ax.text(7.5, 5.0, 'Hidden state = memory\nthat updates each step.', ha='center', va='center', fontsize=10)

# Arrow
ax.annotate('', xy=(11, 5.25), xytext=(9.7, 5.25),
            arrowprops=dict(arrowstyle='->', lw=3, color='gray'))

# Box 3: The Limitation
rect3 = mpatches.FancyBboxPatch((11, 4.5), 3.5, 1.5, boxstyle='round,pad=0.1',
                                 facecolor='#FEF9E7', edgecolor='#F39C12', linewidth=2)
ax.add_patch(rect3)
ax.text(12.75, 5.6, 'THE LIMITATION', ha='center', va='center', fontsize=12,
        fontweight='bold', color='#F39C12')
ax.text(12.75, 5.0, 'Forgets over long\nsequences. Fix:\nLSTM / GRU', ha='center', va='center', fontsize=10)

# The Formula
rect_formula = mpatches.FancyBboxPatch((2, 2.5), 10, 1.2, boxstyle='round,pad=0.1',
                                       facecolor='#EBF5FB', edgecolor='#2980B9', linewidth=2)
ax.add_patch(rect_formula)
ax.text(7, 3.4, 'THE FORMULA', ha='center', va='center', fontsize=12,
        fontweight='bold', color='#2980B9')
ax.text(7, 2.9, r'$h_t = \tanh(W_{xh} \cdot x_t + W_{hh} \cdot h_{t-1} + b_h)$',
        ha='center', va='center', fontsize=14)

# Key properties
props = [
    (1.5, 1.2, 'Parameter\nSharing', 'Same weights\nat every step', '#8E44AD'),
    (5.0, 1.2, 'Variable\nLength', 'Works for any\nsequence length', '#2980B9'),
    (8.5, 1.2, 'Hidden State\n= Memory', 'Compressed summary\nof all past inputs', '#27AE60'),
    (12.0, 1.2, 'Sequential\nProcessing', 'One step at\na time', '#E74C3C'),
]

for x, y, title, desc, color in props:
    rect = mpatches.FancyBboxPatch((x - 1.3, y - 0.7), 2.6, 1.4, boxstyle='round,pad=0.1',
                                   facecolor='white', edgecolor=color, linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y + 0.25, title, ha='center', va='center', fontsize=10,
            fontweight='bold', color=color)
    ax.text(x, y - 0.25, desc, ha='center', va='center', fontsize=9)

# Bottom text
ax.text(7, -1.0, 'Next up: Notebook 02 -- Backpropagation Through Time (how RNNs LEARN)',
        ha='center', va='center', fontsize=12, fontstyle='italic',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

---

## Test Your Understanding

1. **Why can't a regular feedforward network handle the sentence "dog bites man" differently from "man bites dog"?**
   <details>
   <summary>Click to reveal answer</summary>
   A feedforward network processes each word independently with no notion of order. If it sees "dog", "bites", and "man" as separate inputs (or as a bag of words), both sentences look identical. It has no memory of which word came first, second, or third.
   </details>

2. **What is the hidden state, and why is it initialized to zeros?**
   <details>
   <summary>Click to reveal answer</summary>
   The hidden state is a vector that serves as the RNN's memory -- a compressed summary of everything it has seen so far. It is initialized to zeros because at the start, the RNN hasn't seen any input yet, so it has no memory. (Starting with all zeros means "blank slate".)
   </details>

3. **In the formula h_t = tanh(W_xh * x_t + W_hh * h_{t-1} + b_h), what role does each W play?**
   <details>
   <summary>Click to reveal answer</summary>
   W_xh transforms the current input into something useful for the hidden state ("what does this new input tell me?"). W_hh transforms the previous hidden state ("how should I combine my existing memory with new info?"). Together, they blend old memory and new input.
   </details>

4. **Why is parameter sharing important? What would happen without it?**
   <details>
   <summary>Click to reveal answer</summary>
   Parameter sharing means the same weights are used at every time step. Without it, you'd need separate weights for each position (like a feedforward net), which means: (a) the model can't handle sequences longer than what it was built for, (b) it needs far more parameters, and (c) it can't generalize a pattern learned at position 3 to position 30.
   </details>

5. **Why do vanilla RNNs struggle with long sequences? What's the name of this problem?**
   <details>
   <summary>Click to reveal answer</summary>
   The vanishing gradient problem. As the hidden state is transformed many times (once per time step), information from early inputs gets diluted and eventually lost. During training, the gradient signal that should update weights based on early inputs shrinks exponentially. This is why LSTMs and GRUs were invented -- they add gates to preserve information over long distances.
   </details>

---

## What's Next?

Great work! You now understand the fundamentals of RNNs -- the hidden state, unrolling, parameter sharing, and why it all matters.

But we skipped something crucial: **how does an RNN actually learn?** We built the forward pass (feeding data through the network), but we never trained any weights. The RNN needs to learn good values for $W_{xh}$, $W_{hh}$, and $b_h$ so that its predictions actually make sense.

In the next notebook, we'll cover:

- **Backpropagation Through Time (BPTT)** -- the algorithm for computing gradients in RNNs
- Why gradients **vanish** (and sometimes **explode**) over long sequences
- **Gradient clipping** -- a simple trick to prevent explosions
- Why this problem motivated the invention of LSTM and GRU cells

**Ready to learn how RNNs train?** --> [Continue to Notebook 02: Backpropagation Through Time](02_bptt.ipynb)

---

*Well done on completing Notebook 01! You've built an RNN cell from scratch -- most people only ever call `nn.RNN()` in a framework. You actually understand what's inside.*